Exercise 4: KV Caching for Frame-Autoregressive Inference



[← Back to reading list]()

# Exercise 4: KV Caching for Frame-Autoregressive Inference

> **Learning objectives:**

**Learning objectives:**


- Understand why naive frame-autoregressive generation is slow and how KV caching helps

- Adapt the KV-Cache from Exercise 1 to work with the video DiT from Exercise 3

- Handle the subtlety that the *new frame* is denoised over multiple Euler steps (K,V for the new frame change each step, but K,V for previous frames are fixed)

- Benchmark the speedup from KV caching

**Prerequisites:** [Exercise 1]() (KV-Cache basics) and [Exercise 3]() (frame-autoregressive Pong model). Reference implementation: [wendlerc/toy-wm]().

## Background: Why Cache for Video?

Recall from Exercise 3 that video sampling works like this:


- For each new frame, run \(N\) denoising steps (e.g. 30 Euler steps)

- At each denoising step, the model processes **all previous frames + the current noisy frame**

- After denoising, the clean frame becomes context for the next frame

Without caching, generating frame \(f\) requires processing \(f \times P\) tokens at each denoising step (where \(P\) = patches per frame). For a 100-frame video with 36 patches per frame, the last frame requires processing 3600 tokens × 30 denoising steps. The total compute across all frames is enormous.

With KV caching, we only process the **new frame's patches** (\(P\) tokens) at each denoising step, retrieving cached K, V for all previous frames. This gives a roughly \(\sim f\times\) speedup for frame \(f\).

## The Subtlety: What Can Be Cached?

Unlike standard LLM generation (Exercise 1), video denoising has a subtlety:


- **Previous frames** (fully denoised, \(t=0\)): their K, V are fixed. Cache them once when each frame is finalized.

- **Current frame** (being denoised): its K, V change at every Euler step as the frame gets cleaner. These should **not** be cached across denoising steps.

So the strategy is:


- When a frame is fully denoised, run one forward pass for just that frame's tokens and cache their K, V.

- During denoising of the next frame, use the cache for all previous frames and compute fresh K, V for the current frame only.

## Setup

Start from your Exercise 3 code. You'll need all the model components (`RMSNorm`, `MLP`, `VideoPatch`, `VideoUnPatch`, etc.) and a trained `CausalDiT`.

In [ ]:
import torch
import torch as t
from torch import nn
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
# Load your model components from Exercise 3

---

### Exercise 1 — Adapt KVCache for Video

Difficulty: 🔴🔴⭕⭕⭕  |  Importance: 🔵🔵🔵🔵⭕  |  ~10 min

Reuse the `KVCache` from Exercise 1. Add a method to **append** finalized K,V (for a denoised frame) without returning concatenated results, and a method to **get** the cached K,V for use during denoising.

In [ ]:
class VideoKVCache:
    """KV Cache for frame-autoregressive video generation.

    Stores K, V for all finalized (fully denoised) frames.
    During denoising of a new frame, cached K,V are concatenated
    with the new frame's K,V.
    """
    def __init__(self, n_layers):
        # YOUR CODE HERE
        pass

    def append(self, layer_idx, keys, values):
        """Append finalized K,V for a newly denoised frame.

        Args:
            layer_idx: which layer
            keys: (B, n_head, P, d_head) - keys for one frame's patches
            values: (B, n_head, P, d_head)
        """
        # YOUR CODE HERE
        pass

    def get_cached_kv(self, layer_idx):
        """Get all cached K,V for use in attention.

        Returns:
            (cached_keys, cached_values) or (None, None) if empty
        """
        # YOUR CODE HERE
        pass

    def get_and_extend(self, layer_idx, new_keys, new_values):
        """Get cached K,V concatenated with new K,V.

        Used during denoising: cached K,V for previous frames +
        new K,V for the frame being denoised.

        Returns:
            (all_keys, all_values)
        """
        # YOUR CODE HERE
        pass

    @property
    def cached_seq_len(self):
        """Total number of cached tokens (across all finalized frames)."""
        # YOUR CODE HERE
        pass

---

### Exercise 2 — Modify Attention for Cached Video Inference

Difficulty: 🔴🔴🔴🔴⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~25 min

Modify the video attention to use the KV cache. When a cache is provided:


- Only the *new* tokens' Q, K, V are computed from the input

- K, V are concatenated with cached K, V from previous frames

- Q attends to all K's (cached + new) — no mask needed since all cached tokens are from earlier frames (already causally valid)

In [ ]:
class CachedVideoAttention(nn.Module):
    def __init__(self, d=64, n_head=4):
        super().__init__()
        self.n_head = n_head
        self.d = d
        self.d_head = d // n_head
        self.QKV = nn.Linear(d, 3 * d, bias=False)
        self.O = nn.Linear(d, d, bias=False)
        self.normq = RMSNorm(self.d_head)
        self.normk = RMSNorm(self.d_head)

    def forward(self, x, mask=None, kv_cache=None, layer_idx=None, cache_mode=None):
        """
        Args:
            x: (B, S, d) input tokens
            mask: optional attention mask (S_q, S_k) for non-cached mode
            kv_cache: optional VideoKVCache
            layer_idx: layer index for cache
            cache_mode: None (no cache), "denoise" (denoising with cache),
                       or "finalize" (store finalized frame to cache)
        """
        b, s, d = x.shape
        q, k, v = self.QKV(x).chunk(3, dim=-1)
        q = q.reshape(b, s, self.n_head, self.d_head)
        k = k.reshape(b, s, self.n_head, self.d_head)
        v = v.reshape(b, s, self.n_head, self.d_head)
        q = self.normq(q)
        k = self.normk(k)

        # Transpose to (b, n_head, s, d_head)
        q = q.permute(0, 2, 1, 3)
        k = k.permute(0, 2, 1, 3)
        v = v.permute(0, 2, 1, 3)

        if kv_cache is not None and cache_mode == "finalize":
            # YOUR CODE HERE
            # Store this frame's K,V in the cache, then return
            # attention output normally (attending to all cached + current)
            pass

        if kv_cache is not None and cache_mode == "denoise":
            # YOUR CODE HERE
            # Concatenate cached K,V with current K,V
            # No mask needed (new tokens attend to all)
            pass

        # Default: no cache, use mask if provided
        # YOUR CODE HERE
        pass

---

### Exercise 3 — Build Cached CausalDiT

Difficulty: 🔴🔴🔴🔴⭕  |  Importance: 🔵🔵🔵🔵🔵  |  ~25 min

Modify the `CausalDiT` to support two modes of forward pass:


- **Normal mode** (no cache): processes full video as before (for training and non-cached inference)

- **Cached mode**: processes only one frame's worth of tokens, using the cache for previous frames

In [ ]:
class CachedCausalDiT(nn.Module):
    def __init__(self, h=24, w=24, n_actions=4, in_channels=3,
                 patch_size=4, n_blocks=8, d=64, n_head=4, exp=2, T=1000):
        super().__init__()
        # Same as CausalDiT but use CachedVideoAttention in blocks
        # YOUR CODE HERE
        pass

    def forward(self, x, actions, ts, kv_cache=None, cache_mode=None):
        """
        Args:
            x: (B, T_frames, C, H, W) video frames
            actions: (B, T_frames) actions
            ts: (B, T_frames) timesteps
            kv_cache: optional VideoKVCache
            cache_mode: "denoise" or "finalize" or None
        """
        # YOUR CODE HERE
        # When cache_mode is "denoise" or "finalize":
        #   - x should contain only the new frame(s)
        #   - Positional embedding uses pe (no need to tile for multiple frames)
        #   - conditioning is computed for new frame(s) only
        #   - Pass cache_mode and kv_cache through to attention
        # When cache_mode is None:
        #   - Standard full forward pass as before
        pass

---

### Exercise 4 — Implement Cached Video Sampling

Difficulty: 🔴🔴🔴🔴🔴  |  Importance: 🔵🔵🔵🔵🔵  |  ~30 min

This is the capstone exercise. Implement frame-autoregressive sampling with KV caching. The algorithm is:


- **Finalize the first frame:** run forward with `cache_mode="finalize"` to store K,V for the context frame

- **For each new frame:**


- Initialize the new frame as noise

- Run \(N\) denoising steps, each time forwarding only the new frame's tokens with `cache_mode="denoise"`

- After denoising is complete, run one final forward with `cache_mode="finalize"` to store the clean frame's K,V in the cache

In [ ]:
@t.no_grad()
def sample_video_cached(model, first_frame, actions, n_denoise_steps=30, cfg=0):
    """Generate video with KV-cached frame-autoregressive sampling.

    Args:
        model: CachedCausalDiT
        first_frame: (B, 1, C, H, W) context frame
        actions: (B, total_frames) action sequence
        n_denoise_steps: Euler steps per frame
        cfg: classifier-free guidance strength

    Returns:
        (B, total_frames, C, H, W) generated video
    """
    B = first_frame.shape[0]
    total_frames = actions.shape[1]
    C, H, W = first_frame.shape[2:]

    # YOUR CODE HERE
    # 1. Create two KV caches (one for conditional, one for unconditional if cfg > 0)
    # 2. Finalize the first frame
    # 3. For each subsequent frame:
    #    a. Initialize noise
    #    b. Denoise with Euler steps (using cache)
    #    c. Finalize the denoised frame (add to cache)
    # 4. Return the full video
    pass

---

### Exercise 5 — Verify Correctness and Benchmark

Difficulty: 🔴🔴🔴⭕⭕  |  Importance: 🔵🔵🔵🔵⭕  |  ~15 min

Verify that the cached and uncached versions produce the same output (when using the same noise), then benchmark the speedup.

In [ ]:
# Step 1: Correctness check
# YOUR CODE HERE
# - Set a fixed random seed
# - Generate a short video (e.g. 5 frames) with sample_video (uncached)
# - Reset the seed to the same value
# - Generate the same video with sample_video_cached
# - Assert the outputs match (t.allclose with reasonable tolerance)

# Step 2: Benchmark
# YOUR CODE HERE
# - Time both methods for generating 10, 20, 50 frames
# - Plot the speedup as a function of video length

> **Note:**
**Expected results:** You should see speedups that grow roughly linearly with the number of frames. For 20 frames, expect a ~5–10x speedup. For longer sequences (50+ frames), the speedup becomes even more dramatic. This is because without caching, each denoising step of frame \(f\) processes \(f \times P\) tokens, while with caching it always processes just \(P\) tokens.

## Bonus: Sliding Window Cache

For very long videos, even the KV cache grows large. A **sliding window** approach keeps only the most recent \(W\) frames in the cache, discarding older ones. This trades off long-range temporal coherence for bounded memory usage. Check the [toy-wm reference implementation]() for an example.

## Summary

Congratulations! You've built a complete pipeline from scratch:


- **Exercise 1:** KV-Caching fundamentals for autoregressive transformers

- **Exercise 2:** Flow matching for image generation (patchify, DiT, training, sampling, CFG)

- **Exercise 3:** Frame-autoregressive video generation (temporal extension, causal attention, diffusion forcing)

- **Exercise 4:** KV-caching for efficient video inference

These are the core building blocks behind modern frame-autoregressive video generation and world models. The same principles scale to much larger models, higher resolutions, and more complex environments.

[← Previous: Exercise 3]()   [Back to reading list]()